# Notebook 04 — Standard STGCN

## Objective
Implement a Spatio-Temporal Graph Convolutional Network as a graph-based traffic forecasting baseline.

## Core Idea
Traffic forecasting involves two types of dependencies:

1. Temporal dependencies: how traffic changes over time.
2. Spatial dependencies: how measurements at connected sensors relate to one another.

STGCN combines temporal modelling and graph-based spatial information.

## Graph Representation
- Nodes represent traffic sensors.
- Edges represent relationships between sensors.
- The adjacency matrix encodes the graph structure used by the model.

## Model Workflow
1. Receive historical traffic sequences.
2. Extract temporal patterns.
3. Aggregate information from connected sensors using graph operations.
4. Produce predictions for the future time steps.

## Previously Recorded Result
- MAE: 6.2633
- RMSE: 11.0173
- MAPE: 13.1616%

Verify this result against the final checkpoint and evaluation output.

## Why This Model Matters
The standard STGCN provides a reference point for assessing whether later architectural modifications offer meaningful improvements.

## Review Reminder
Be prepared to explain:
- Why traffic sensors are represented as graph nodes.
- What the adjacency matrix represents.
- The difference between temporal convolution and graph convolution.
- Why spatial and temporal information are both useful.

# Phase 4: STGCN Model

In [1]:
import sys
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if "google.colab" in sys.modules:
    if "/content" not in sys.path:
        sys.path.insert(0, "/content")
else:
    sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

from project_paths import *

In [2]:
def load_npz(path):
    data = np.load(path)
    return data['X'], data['Y']

x_train, y_train = load_npz(PROCESSED_DATA_DIR / "train" / "sequences.npz")
x_val, y_val = load_npz(PROCESSED_DATA_DIR / "validation" / "sequences.npz")
x_test, y_test = load_npz(PROCESSED_DATA_DIR / "test" / "sequences.npz")

print(f"Train shape: {x_train.shape} -> {y_train.shape}")
print(f"Val shape: {x_val.shape} -> {y_val.shape}")
print(f"Test shape: {x_test.shape} -> {y_test.shape}")

history_steps = x_test.shape[1]
horizon_steps = y_test.shape[1]
nodes = x_test.shape[2]
features = x_test.shape[3]

print(f"\nExtracted Config:")
print(f"History: {history_steps}")
print(f"Horizon: {horizon_steps}")
print(f"Nodes: {nodes}")
print(f"Features: {features}")

Train shape: (23976, 12, 207, 1) -> (23976, 3, 207, 1)
Val shape: (5126, 12, 207, 1) -> (5126, 3, 207, 1)
Test shape: (5128, 12, 207, 1) -> (5128, 3, 207, 1)

Extracted Config:
History: 12
Horizon: 3
Nodes: 207
Features: 1


In [3]:
from src.data.dataset import get_dataloaders

batch_size = 64
train_loader, val_loader, test_loader = get_dataloaders(
    x_train, y_train,
    x_val, y_val,
    x_test, y_test,
    batch_size=batch_size
)

batch_x, batch_y = next(iter(test_loader))
print(f"Batch X: {batch_x.shape}")
print(f"Batch Y: {batch_y.shape}")

Batch X: torch.Size([64, 12, 207, 1])
Batch Y: torch.Size([64, 3, 207, 1])


## Graph Loading

In [4]:
from src.data.graph import load_adjacency_matrix, calculate_scaled_laplacian, calculate_cheb_poly

adj_mx = load_adjacency_matrix(ADJACENCY_FILE)
L_tilde = calculate_scaled_laplacian(adj_mx)
Ks = 3
cheb_tensor = calculate_cheb_poly(L_tilde, Ks)
print("Cheb Tensor Shape:", cheb_tensor.shape)

Cheb Tensor Shape: torch.Size([3, 207, 207])


In [5]:
from src.models.stgcn import STGCN
from src.training.train import train_model
from src.evaluation.evaluate import evaluate_model_comprehensive
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cheb_tensor = cheb_tensor.to(device)

blocks = [[64, 16, 3], [16, 64, 3]]
stgcn_model = STGCN(blocks, n_nodes=nodes, history_steps=history_steps, horizon_steps=horizon_steps, features=features, Ks=Ks)
optimizer = torch.optim.Adam(stgcn_model.parameters(), lr=0.001)
loss_fn = nn.L1Loss()

In [6]:
# Using a custom loop for STGCN to pass cheb_polynomials
num_epochs = 3
stgcn_model.to(device)

for epoch in range(num_epochs):
    stgcn_model.train()
    train_loss = 0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        preds = stgcn_model(batch_x, cheb_tensor)
        loss = loss_fn(preds, batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * batch_x.size(0)
    print(f"Epoch {epoch+1} Train Loss: {train_loss/len(train_loader.dataset):.4f}")

Epoch 1 Train Loss: 0.2116


Epoch 2 Train Loss: 0.1777


Epoch 3 Train Loss: 0.1760


In [7]:
results = evaluate_model_comprehensive(stgcn_model, test_loader, device, stgcn_cheb=cheb_tensor)
print("STGCN Metrics:", results['overall'])
model_path = PROJECT_ROOT / 'experiments' / 'stgcn_weights.pt'
torch.save(stgcn_model.state_dict(), model_path)
print(f"Saved weights to {model_path}")

STGCN Metrics: {'MAE': 0.17729486525058746, 'RMSE': 0.42537736892700195, 'MAPE': 0.11438785493373871, 'WAPE': 0.21419280767440796, 'R2': 0.8760931491851807}
Saved weights to /Users/sushank/Documents/GitHub/urbanAi/experiments/stgcn_weights.pt
